<span STYLE="font-size:150%"> 
    Segment microCT scans
</span>

Docker image: gnasello/slicer-env:2023-07-06 \
Latest update: 10 March 2023

- load image stack in Slicer
- segment mineralized tissue
- compute segmented statistics (volumes)

# Load libraries

In [ ]:
import pyslicer as ps
import slicer
from pathlib import Path
import pandas as pd
import pyvista as pv

# Volume input

## Load `.nrrd` file

Write the path of the `.nrrd` file and load it to Slicer

In [ ]:
# this cell is tagged 'parameters'
volume_file = 'microCT_volume/microCT_volume_preview.nrrd'
segment_file = 'segmented_volumes/Bone.seg.nrrd'
output_dir_path = 'segmented_volumes'
directory_notebook = Path().parent.absolute()
sample_name = directory_notebook.stem

In [ ]:
# # Parameters
# volume_file = "/config/researcher_home/Documents/microCT/2026-03-24_TV-GN_blank_gels/043609/microCT_volume/microCT_volume_preview.nrrd"
# segment_file = "/config/researcher_home/Documents/microCT/2026-03-24_TV-GN_blank_gels/043609/segmented_volumes/Bone.seg.nrrd"
# output_dir_path = "/config/researcher_home/Documents/microCT/2026-03-24_TV-GN_blank_gels/043609/segmented_volumes"


In [ ]:
file_path = Path(volume_file)

# Remove image numbering _0000, _0001 ...
filename_output = file_path.stem[:-4]

In [ ]:
masterVolumeNode = slicer.util.loadNodeFromFile(file_path.resolve())

Print spacing

In [ ]:
## mm
masterVolumeNode.GetSpacing()

Make ```segmented_volumes``` folder

In [ ]:
output_directory = Path(output_dir_path)

output_directory.mkdir(parents=True, exist_ok=True)

In [ ]:
# Monitor Memory in Slicer
import psutil, os
print(psutil.Process(os.getpid()).memory_info().rss / (1024**3), "GB used")

# Create segmentationNode

## Create segmentation-related nodes

Create segmentation node

In [ ]:
file_path = Path(segment_file)

segmentationNode = slicer.util.loadSegmentation(file_path.resolve(), properties={'name':"Segmentation"})

Create temporary segment editor to get access to effects

In [ ]:
segmentEditorWidget, segmentEditorNode = ps.segmentation.segmentEditorWidget(segmentationNode = segmentationNode, 
                                                                             masterVolumeNode = masterVolumeNode)

# Operation on segments

## Manual fix of the segmentation

Sometimes it might be necessary to remove speckles at the image boundaries. If so, use the `scissor` tool in the `Segment Editor` before proceeding with the rest of the script. 

## Split islands

SPLIT_ISLANDS operation from the [SegmentEditorIslandsEffect](https://github.com/Slicer/Slicer/blob/294ef47edbac2ccb194d5ee982a493696795cdc0/Modules/Loadable/Segmentations/EditorEffects/Python/SegmentEditorIslandsEffect.py#L402)

In [ ]:
segment_name = 'Segment_1'
minimum_size = 20000 #number of voxels

In [ ]:
ps.segmentation.split_islands(minimum_size, 
                              segment_name, 
                              segmentEditorNode, 
                              segmentEditorWidget)

# Export model as .stl file

## Convert all segments to model nodes

Get closed surface representation of the segment, from [slicer scripting repository](https://slicer.readthedocs.io/en/latest/developer_guide/script_repository.html#export-nodes-warped-by-transform-sequence)

In [ ]:
segmentationNode.CreateClosedSurfaceRepresentation()

In [ ]:
shNode = slicer.mrmlScene.GetSubjectHierarchyNode()
exportFolderItemId = shNode.CreateFolderItem(shNode.GetSceneItemID(), "Segments")
slicer.modules.segmentations.logic().ExportAllSegmentsToModels(segmentationNode, exportFolderItemId)

In [ ]:
import slicer
import vtk
import numpy as np

# ---------------------------------------------------------------------
# Get model nodes
# ---------------------------------------------------------------------

modelA = slicer.util.getNode("Segment_1")
modelB = slicer.util.getNode("Segment_1_2")

polyA = modelA.GetPolyData()
polyB = modelB.GetPolyData()

# ---------------------------------------------------------------------
# Build fast spatial locator on B (triangle-based)
# ---------------------------------------------------------------------

locatorB = vtk.vtkStaticCellLocator()
locatorB.SetDataSet(polyB)
locatorB.BuildLocator()

locatorA = vtk.vtkStaticCellLocator()
locatorA.SetDataSet(polyA)
locatorA.BuildLocator()

# ---------------------------------------------------------------------
# Helper: A -> B
# ---------------------------------------------------------------------

def surface_min_distance(sourcePoly, targetLocator):

    minDist2 = float("inf")
    bestA = None
    bestB = None

    pA = [0.0, 0.0, 0.0]
    closest = [0.0, 0.0, 0.0]

    cellId = vtk.mutable(0)
    subId = vtk.mutable(0)

    for i in range(sourcePoly.GetNumberOfPoints()):

        sourcePoly.GetPoint(i, pA)

        dist2_ref = vtk.mutable(0.0)

        targetLocator.FindClosestPoint(
            pA,
            closest,
            cellId,
            subId,
            dist2_ref
        )

        # convert VTK reference → Python float
        dist2 = float(dist2_ref)

        if dist2 < minDist2:
            minDist2 = dist2
            bestA = np.array(pA)
            bestB = np.array(closest)

    return np.sqrt(minDist2), bestA, bestB

# ---------------------------------------------------------------------
# Compute both directions
# ---------------------------------------------------------------------

distAB, A1, B1 = surface_min_distance(polyA, locatorB)
distBA, B2, A2 = surface_min_distance(polyB, locatorA)

# ---------------------------------------------------------------------
# Pick global minimum
# ---------------------------------------------------------------------

if distAB <= distBA:
    minDist = distAB
    pA, pB = A1, B1
else:
    minDist = distBA
    pA, pB = A2, B2

# ---------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------

print("Minimum surface distance:", minDist)
print("Point on A:", pA)
print("Point on B:", pB)

In [ ]:
# Create fiducials
# fidA = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsFiducialNode", "ClosestPoint_A")
# fidA.AddControlPoint(pA)

# fidB = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsFiducialNode", "ClosestPoint_B")
# fidB.AddControlPoint(pB)

# Create line
lineNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsLineNode", "Distance")
lineNode.AddControlPoint(pA)
lineNode.AddControlPoint(pB)

In [ ]:
import csv
import os

output_path = output_directory / "min_distance_result.csv"

with open(output_path, mode="w", newline="") as f:
    writer = csv.writer(f)

    # header
    writer.writerow([
        "Sample",
        "min_distance",
        "pA_x", "pA_y", "pA_z",
        "pB_x", "pB_y", "pB_z"
    ])

    # data row
    writer.writerow([
        sample_name,
        minDist,
        pA[0], pA[1], pA[2],
        pB[0], pB[1], pB[2]
    ])

print("Saved to:", output_path)